# Aerodynamics at the FIFA World Cup

## Introduction

Let's take a look at how aerodynamic forces shape athletic performance by building some computational fluid dynamics simulations.

#### Contents

1. **Background Theory**: Briefly explain the Navier-Stokes equation.

2. **The Knuckleball (No-Spin Wake) vs a Regular Shot (Spin Wake)**: To explain the erratic trajectory of a knuckleball, a side-by-side comparison of a rotating and non-rotating cylinder reveals how spin stabilises the wake, delays flow separation, and alters the pressure field.

3. **Tactical Positioning**: Velocity and pressure fields around single and multiple athlete silhouettes reveal the drag-reduction benefits of specific positioning tactics and in-game situations.

Built with [ΦFlow](https://github.com/tum-pbs/PhiFlow) (v3.4+, JAX backend), a differentiable PDE solver on Cartesian grids.

## 1. Background Theory

### Incompressible Navier–Stokes

$$
\rho \left(\frac{\partial \mathbf{u}}{\partial t} + \mathbf{u} \cdot \nabla \mathbf{u}\right) = -\nabla p + \mu \nabla^2 \mathbf{u} + \mathbf{f}
$$

### Continuity (incompressibility constraint)

$$
\nabla \cdot \mathbf{u} = 0
$$

| Symbol | Quantity | Units |
|--------|----------|-------|
| $\mathbf{u}$ | Velocity field | m/s |
| $p$ | Pressure | Pa |
| $\rho$ | Density (air ≈ 1.225) | kg/m³ |
| $\mu$ | Dynamic viscosity (air ≈ 1.81×10⁻⁵) | Pa·s |
| $\mathbf{f}$ | Body forces (buoyancy, etc.) | N/m³ |

### Key dimensionless group

$$
Re = \frac{\rho U L}{\mu}
$$

For a soccer ball ($L \approx 0.22$ m) at $U \approx 30$ m/s:
$Re \approx 4.5 \times 10^5$ — well into the turbulent regime. Our 2D simulation at $Re \approx 10^3$ captures the essential wake physics at a feasible grid resolution.

Talk about why we built a simple dimensionless solver.

In [ ]:
"""Imports and ΦFlow backend configuration."""
from phi.jax.flow import *
from phi import geom, math, field

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from matplotlib.patches import Circle
from matplotlib.patches import Ellipse
from pathlib import Path
from tqdm.notebook import trange
from IPython.display import HTML
import os, sys

# ── Dark theme ──
plt.style.use("dark_background")
matplotlib.rcParams.update(
    {
        "figure.facecolor": "#111111",
        "axes.facecolor": "#111111",
        "axes.edgecolor": "#cccccc",
        "axes.labelcolor": "#ffffff",
        "text.color": "#ffffff",
        "xtick.color": "#cccccc",
        "ytick.color": "#cccccc",
    }
)

# ── Output directory ──
ASSETS = Path("assets")
ASSETS.mkdir(exist_ok=True)

## 2. The Aerodynamics of a Shot

### Magnus Effect vs Knuckleball Effect

#### The Physics of a Spinning Ball (Magnus)

When a soccer ball spins during flight, the rotation drags the boundary layer around the surface via viscosity. On the side where the surface motion opposes the freestream flow, the boundary layer decelerates and separates early. On the side where surface motion aligns with the flow, the boundary layer is energised and separation is delayed. This asymmetric separation creates a pressure imbalance: lower pressure on the delayed-separation side and higher on the early-separation side, producing a net **lift force** perpendicular to the flight path (the Magnus effect). This is what bends a free kick around a wall.

#### The Physics of a Non-Spinning Ball (Knuckleball)

A non-spinning (or very slow-spinning) ball has no preferred asymmetry, so its boundary layer separates symmetrically on both sides. However, the separated shear layers roll up into an alternating **Kármán vortex street**, vortices shed first from one side, then the other, in a periodic pattern. This unsteady wake produces an oscillating lateral force and a wider, more turbulent wake than the spinning case. The resulting trajectory becomes unpredictable and "fluttering," the hallmark of a knuckleball.

#### Aerodynamic Concepts Illustrated

| Concept | Meaning | Seen In |
|---------|---------|---------|
| **Boundary layer separation** | The point where flow detaches from the surface, creating a wake | Streamlines leaving the cylinder surface |
| **Wake** | Region of recirculating, low-velocity flow behind the body | Dark (slow) region in velocity plots |
| **Pressure drag** | Pressure difference between the front (stagnation) and rear (wake) of the body | Colour contrast in pressure plots |
| **Vortex shedding** | Alternating vortices peeled from each side of the body | Wavy, unsteady streamlines in the knuckleball case |
| **Magnus lift** | Lateral force from asymmetric pressure field | Asymmetric colour pattern in Magnus pressure plot |

#### What the Visualisations Show

**Velocity magnitude** (inferno colormap): Dark = slow (wake), bright = fast (flow accelerates around the cylinder). The Magnus case shows a narrower, deflected wake; the knuckleball case shows a wider, symmetric, unsteady wake.

**Pressure field** (magma colormap): High pressure at the front stagnation point (yellow), low pressure in the separated wake region (dark). In the Magnus case the low-pressure region is asymmetric, indicating a net lift force.

**Streamlines** trace the instantaneous flow direction. Vortices appear as closed loops behind the cylinder. Compare the regular shedding pattern (knuckleball) vs the stabilised, deflected wake (Magnus).

#### Setup

- **Grid**: 256 × 128 cells, 8 × 4 unit domain (fine streamwise resolution)
- **Cylinder**: `geom.Sphere` radius 0.3 at (2.0, 2.0) — 2D cross-section of a ball
- **Left panel (Magnus)**: angular velocity 10 rad/s
- **Right panel (Knuckleball)**: angular velocity 0 rad/s
- **Inflow**: uniform 1.0 m/s from left boundary (Re ≈ 10³,
  captures essential wake physics at feasible grid resolution)
- **Solver**: ΦFlow incompressible Navier-Stokes, 120 steps × Δt = 0.3

In [ ]:
"""Module 2 — Run spinning vs non-spinning cylinder simulations."""

# ── Domain & boundary conditions (same as Module 1) ──
BOX = geom.Box(x=8.0, y=4.0)
BOUNDARY = {
    "x-": vec(x=1.0, y=0.0),  # inflow
    "x+": ZERO_GRADIENT,  # outflow
    "y": 0,  # no-slip walls
}

# ── Cylinder geometry ──
CENTER = (2.0, 2.0)
RADIUS = 0.3
CYLINDER = geom.Sphere(x=2.0, y=2.0, radius=0.3)

# ── Two obstacles: spinning and non-spinning ──
obstacle_spin = Obstacle(CYLINDER, velocity=vec(x=0.0, y=0.0), angular_velocity=10.0)
obstacle_nospin = Obstacle(CYLINDER, velocity=vec(x=0.0, y=0.0), angular_velocity=0.0)

# ── Run parameters ──
DT = 0.3
N_STEPS = 160
SAVE_EVERY = 4
_centered = CenteredGrid(0, x=256, y=128, bounds=BOX)


def run_cylinder(obstacle, label):
    v0 = StaggeredGrid((1.0, 0.0), BOUNDARY, x=256, y=128, bounds=BOX)
    v, _ = fluid.make_incompressible(v0, ())
    frames = []
    v = v0
    for i in trange(N_STEPS, desc=label):
        v = advect.semi_lagrangian(v, v, dt=DT)
        v, p = fluid.make_incompressible(v, obstacle)
        if i % SAVE_EVERY == 0:
            vc = v @ _centered
            u_c = np.array(vc.vector[0].values.native("x", "y"))
            v_c = np.array(vc.vector[1].values.native("x", "y"))
            vel_mag = field.vec_length(v)
            p_np = np.array(p.values.native("x", "y"))
            frames.append(
                {
                    "pressure": p_np,
                    "u": u_c,
                    "v": v_c,
                    "velocity": np.array(vel_mag.values.native("x", "y")),
                    "step": i,
                }
            )
    return frames


print("Magnus (\u03c9 = 10 rad/s)")
frames_magnus = run_cylinder(obstacle_spin, "Magnus")

print("\nKnuckleball (\u03c9 = 0 rad/s)")
frames_knuckle = run_cylinder(obstacle_nospin, "Knuckleball")

print(f"\nCaptured {len(frames_magnus)} frames each")
print(f"  u shape: {frames_magnus[0]['u'].shape}")

In [ ]:
"""Module 2 — Two-panel knuckleball vs Magnus comparison animation."""

n_frames = min(len(frames_magnus), len(frames_knuckle))
fm = frames_magnus[:n_frames]
fk = frames_knuckle[:n_frames]

x_s = np.linspace(0, 8.0, 256)
y_s = np.linspace(0, 4.0, 128)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor="#111111")
fig.suptitle(
    "Knuckleball vs Regular Shot — Wake Comparison",
    color="white",
    fontsize=14,
    y=0.98,
)

titles = [
    "Magnus Effect (\u03c9 = 10 rad/s)",
    "Knuckleball (\u03c9 = 0 rad/s)",
]
frames_list = [fm, fk]
imgs = []

for ax, title, frames in zip([ax1, ax2], titles, frames_list):
    ax.set_facecolor("#111111")
    ax.set_title(title, color="white", fontsize=11)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0.5, 7.5)
    ax.set_ylim(0.5, 3.5)

    im = ax.imshow(
        frames[0]["velocity"].T,
        origin="lower",
        cmap="inferno",
        aspect="auto",
        extent=[0, 8, 0, 4],
        vmin=0,
        vmax=1.5,
    )
    ax.add_patch(Circle(CENTER, RADIUS, color="white", fill=False, lw=2.0))
    s = ax.streamplot(
        x_s,
        y_s,
        frames[0]["u"].T,
        frames[0]["v"].T,
        color=frames[0]["velocity"].T,
        cmap="inferno",
        linewidth=0.8,
        density=1.5,
        arrowstyle="->",
        arrowsize=0.6,
    )
    imgs.append(im)

fig.tight_layout(rect=[0, 0, 1, 0.93])
cbar = fig.colorbar(imgs[0], ax=[ax1, ax2], fraction=0.04, pad=0.02)
cbar.set_label("Velocity magnitude", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")


def update(idx):
    for ax, frames in zip([ax1, ax2], frames_list):
        ax.clear()
        ax.set_facecolor("#111111")
        ax.set_xlim(0.5, 7.5)
        ax.set_ylim(0.5, 3.5)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        im = ax.imshow(
            frames[idx]["velocity"].T,
            origin="lower",
            cmap="inferno",
            aspect="auto",
            extent=[0, 8, 0, 4],
            vmin=0,
            vmax=1.5,
        )
        ax.add_patch(Circle(CENTER, RADIUS, color="white", fill=False, lw=2.0))
        s = ax.streamplot(
            x_s,
            y_s,
            frames[idx]["u"].T,
            frames[idx]["v"].T,
            color=frames[idx]["velocity"].T,
            cmap="inferno",
            linewidth=0.8,
            density=1.5,
            arrowstyle="->",
            arrowsize=0.6,
        )
    return (ax1, ax2)


anim = FuncAnimation(fig, update, frames=n_frames, interval=80, blit=False)

path = ASSETS / "knuckleball_comparison.mp4"
anim.save(str(path), writer="ffmpeg", fps=10, dpi=150)
print(f"Saved {path}")

HTML(anim.to_jshtml())

In [ ]:
"""Module 2 — Two-panel pressure field comparison. Export to MP4."""

n_frames = min(len(frames_magnus), len(frames_knuckle))
fm = frames_magnus[:n_frames]
fk = frames_knuckle[:n_frames]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor="#111111")
fig.suptitle(
    "Knuckleball vs Magnus — Pressure Field",
    color="white",
    fontsize=14,
    y=0.98,
)

titles = [
    "Magnus Effect (\u03c9 = 10 rad/s)",
    "Knuckleball (\u03c9 = 0 rad/s)",
]
frames_list = [fm, fk]
imgs = []

for ax, title, frames in zip([ax1, ax2], titles, frames_list):
    ax.set_facecolor("#111111")
    ax.set_title(title, color="white", fontsize=11)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0.5, 7.5)
    ax.set_ylim(0.5, 3.5)

    p0 = frames[0]["pressure"]
    im = ax.imshow(
        p0.T,
        origin="lower",
        cmap="magma",
        aspect="auto",
        extent=[0, 8, 0, 4],
    )
    ax.add_patch(Circle(CENTER, RADIUS, color="white", fill=False, lw=2.0))
    imgs.append(im)

fig.tight_layout(rect=[0, 0, 1, 0.93])
cbar = fig.colorbar(imgs[0], ax=[ax1, ax2], fraction=0.04, pad=0.02)
cbar.set_label("Pressure", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")


def update(idx):
    for ax, frames, im in zip([ax1, ax2], frames_list, imgs):
        data = frames[idx]["pressure"]
        im.set_data(data.T)
        im.set_clim(data.min(), data.max())
    fig.suptitle(
        f"Knuckleball vs Magnus — Step {frames_list[0][idx]['step']}"
        f"  (t = {frames_list[0][idx]['step'] * 0.3:.1f}s)",
        color="white",
        fontsize=14,
        y=0.98,
    )
    return imgs


anim = FuncAnimation(fig, update, frames=n_frames, interval=80, blit=True)

path = ASSETS / "knuckleball_pressure.mp4"
anim.save(str(path), writer="ffmpeg", fps=10, dpi=150)
print(f"Saved {path}")

HTML(anim.to_jshtml())

# Module 3: The Peloton & Running Pack

## Wake Interaction & Drafting Efficiency

Drag accounts for up to 80% of the resistive force on a runner at sprint speeds. By positioning behind another athlete, a runner enters their **wake** — a region of reduced dynamic pressure — lowering the drag they must overcome.

We study three configurations with **rectangular runner cross-sections**
(0.3 × 0.5 units, rotated torso profile):

| Case | Setup | Expected Effect |
|------|-------|----------------|
| **A** — Solo | Single runner (baseline) | Reference drag |
| **B** — Inline Drafting | Two runners on same line, 1.0 unit apart | Trailing runner sees reduced drag |
| **C** — Echelon Offset | Two runners offset diagonally, 0.6 × 0.3 gap | Partial wake coverage |

### Setup

- **Grid**: 128 × 256, 4 × 8 unit domain, uniform inflow 1.0 m/s
- **Runner shape**: `geom.Box` — 0.3 wide × 0.5 tall
- **Solver**: ΦFlow incompressible Navier-Stokes, 80 steps × Δt = 0.3

In [ ]:
"""Module 3 — Three wake cases with rectangular runners and face-based drag."""

# ── Domain ───────────────────────────────────────────────────────────────────
BOX = geom.Box(x=8.0, y=4.0)
BOUNDARY = {"x-": vec(x=1.0, y=0.0), "x+": ZERO_GRADIENT, "y": 0}

# ── Runner shape ─────────────────────────────────────────────────────────────
HW, HH = 0.15, 0.25  # half-width (x), half-height (y) → 0.3 × 0.5


def runner_at(cx, cy):
    return geom.Box(x=(cx - HW, cx + HW), y=(cy - HH, cy + HH))


# ── Positions (8×4 domain) ──────────────────────────────────────────────────
case_a = [Obstacle(runner_at(2.0, 2.0), velocity=vec(x=0.0, y=0.0))]

case_b = [
    Obstacle(runner_at(2.0, 2.0), velocity=vec(x=0.0, y=0.0)),
    Obstacle(runner_at(3.5, 2.0), velocity=vec(x=0.0, y=0.0)),
]

case_c = [
    Obstacle(runner_at(2.0, 2.0), velocity=vec(x=0.0, y=0.0)),
    Obstacle(runner_at(3.5, 2.35), velocity=vec(x=0.0, y=0.0)),
]

# ── Face-based drag masks ────────────────────────────────────────────────────
x_g = np.linspace(0, 8.0, 256)
y_g = np.linspace(0, 4.0, 128)
XX, YY = np.meshgrid(x_g, y_g, indexing="ij")


def face_drag_mask(cx, cy):
    up = (XX > cx - HW * 1.5) & (XX < cx - HW * 0.8) & (YY > cy - HH) & (YY < cy + HH)
    dn = (XX > cx + HW * 0.8) & (XX < cx + HW * 1.5) & (YY > cy - HH) & (YY < cy + HH)
    return up, dn


def box_drag(p_np, cx, cy):
    up, dn = face_drag_mask(cx, cy)
    return float(p_np[up].mean() - p_np[dn].mean()) if up.sum() and dn.sum() else 0.0


# ── Centered grid for velocity resampling ────────────────────────────────────
_centered = CenteredGrid(0, x=256, y=128, bounds=BOX)

# ── Simulation runner ────────────────────────────────────────────────────────
DT = 0.3
N_STEPS = 160
SAVE_EVERY = 2


def run_case(obstacles, label, drag_pos=None):
    v0 = StaggeredGrid((1.0, 0.0), BOUNDARY, x=256, y=128, bounds=BOX)
    v, _ = fluid.make_incompressible(v0, ())
    frames = []
    drag_series = []
    v = v0

    for i in trange(N_STEPS, desc=label):
        v = advect.semi_lagrangian(v, v, dt=DT)
        v, p = fluid.make_incompressible(v, obstacles)

        if i % SAVE_EVERY == 0:
            vel_mag = field.vec_length(v)
            vel_np = np.array(vel_mag.values.native("x", "y"))
            vc = v @ _centered
            u_c = np.array(vc.values.vector[0].native("x", "y"))
            v_c = np.array(vc.values.vector[1].native("x", "y"))
            frames.append(
                {
                    "velocity": vel_np,
                    "u": u_c,
                    "v": v_c,
                    "step": i,
                }
            )
            if drag_pos:
                drag_series.append(
                    box_drag(np.array(p.values.native("x", "y")), *drag_pos)
                )

    avg_drag = np.mean(drag_series[-len(drag_series) // 5 :]) if drag_series else 0.0
    return frames, avg_drag, drag_series


# ── Execute ──────────────────────────────────────────────────────────────────
print("Case A — Solo (baseline)")
frames_a, drag_a, _ = run_case(case_a, "Case A (solo)")

print("\nCase B — Inline drafting")
frames_b, drag_b, drag_b_s = run_case(case_b, "Case B (inline)", (3.5, 2.0))

print("\nCase C — Echelon offset")
frames_c, drag_c, drag_c_s = run_case(case_c, "Case C (echelon)", (3.5, 2.35))

print(f"\nDrag: A={drag_a:.4f}  B={drag_b:.4f}  C={drag_c:.4f}")
print(f"  B/A = {drag_b / drag_a:.3f}  C/A = {drag_c / drag_a:.3f}")

# ── Print comparison ──
print("\n" + "=" * 50)
print("Drag Comparison (relative to solo)")
print("=" * 50)
print(f"  A — Solo:                      1.000")
print(f"  B — Inline drafting:           {drag_b / drag_a:.3f}")
print(f"  C — Echelon offset:            {drag_c / drag_a:.3f}")
print(f"  Drag reduction (inline):       {(1 - drag_b / drag_a) * 100:.1f}%")
print(f"  Drag reduction (echelon):      {(1 - drag_c / drag_a) * 100:.1f}%")
print("=" * 50)

In [ ]:
# ── Module 3: Three-panel wake comparison — Velocity magnitude ──────────

n_frames = min(len(frames_a), len(frames_b), len(frames_c))
for f in [frames_a, frames_b, frames_c]:
    while len(f) > n_frames:
        f.pop()

runner_sets = [
    [(2.0, 2.0)],
    [(2.0, 2.0), (3.5, 2.0)],
    [(2.0, 2.0), (3.5, 2.35)],
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor="#111111")
fig.suptitle("Wake Comparison — Velocity Magnitude", color="white", fontsize=15, y=0.98)

labels = [
    f"A — Solo Runner (drag = 1.000)",
    f"B — Inline Drafting (drag = {drag_b / drag_a:.3f})",
    f"C — Echelon Offset (drag = {drag_c / drag_a:.3f})",
]

imgs = []
for ax, label, runners, frames in zip(
    axes, labels, runner_sets, [frames_a, frames_b, frames_c]
):
    ax.set_facecolor("#111111")
    ax.set_title(label, color="white", fontsize=10)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 4)
    for cx, cy in runners:
        ax.add_patch(
            Rectangle(
                (cx - HW, cy - HH), 2 * HW, 2 * HH, color="white", fill=False, lw=1.5
            )
        )
    im = ax.imshow(
        frames[0]["velocity"].T,
        origin="lower",
        cmap="inferno",
        aspect="auto",
        extent=[0, 8, 0, 4],
        vmin=0,
        vmax=1.5,
    )
    imgs.append(im)

fig.tight_layout(rect=[0, 0, 1, 0.93])
cbar = fig.colorbar(imgs[0], ax=axes, fraction=0.02, pad=0.02)
cbar.set_label("Velocity magnitude", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")


def update(idx):
    for im, frames in zip(imgs, [frames_a, frames_b, frames_c]):
        im.set_data(frames[idx]["velocity"].T)
    fig.suptitle(
        f"Wake Comparison — Step {frames_a[idx]['step']}  "
        f"(t = {frames_a[idx]['step'] * DT:.1f}s)",
        color="white",
        fontsize=14,
        y=0.98,
    )
    return imgs


anim = FuncAnimation(fig, update, frames=n_frames, interval=80, blit=True)
anim.save(str(ASSETS / "wake_comparison.mp4"), writer="ffmpeg", fps=12, dpi=150)
print("Saved assets/wake_comparison.mp4")

HTML(anim.to_jshtml())

In [ ]:
# ── Module 3: Three-panel wake comparison — Streamlines ─────────────────

x_s = np.linspace(0, 8, 256)
y_s = np.linspace(0, 4, 128)
n_frames = min(len(frames_a), len(frames_b), len(frames_c))

runner_sets = [
    [(2.0, 2.0)],
    [(2.0, 2.0), (3.5, 2.0)],
    [(2.0, 2.0), (3.5, 2.35)],
]

labels = [
    f"A — Solo (drag = 1.000)",
    f"B — Inline (drag = {drag_b / drag_a:.3f})",
    f"C — Echelon (drag = {drag_c / drag_a:.3f})",
]

# ── Static preview ───────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor="#111111")
fig.suptitle("Wake Comparison — Streamlines", color="white", fontsize=15, y=0.98)

for ax, label, runners, frames in zip(
    axes, labels, runner_sets, [frames_a, frames_b, frames_c]
):
    ax.set_facecolor("#111111")
    ax.set_title(label, color="white", fontsize=10)
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 4)
    for cx, cy in runners:
        ax.add_patch(
            Rectangle(
                (cx - HW, cy - HH), 2 * HW, 2 * HH, color="white", fill=False, lw=1.5
            )
        )
    ax.streamplot(
        x_s,
        y_s,
        frames[0]["u"].T,
        frames[0]["v"].T,
        color=frames[0]["velocity"].T,
        cmap="inferno",
        linewidth=0.8,
        density=1.2,
        arrowstyle="->",
        arrowsize=0.5,
    )

fig.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()

# ── Animation ───────────────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(15, 4), facecolor="#111111")
fig2.suptitle("Wake Comparison — Streamlines", color="white", fontsize=15, y=0.98)


def animate(idx):
    for ax, label, runners, frames in zip(
        axes2, labels, runner_sets, [frames_a, frames_b, frames_c]
    ):
        ax.clear()
        ax.set_facecolor("#111111")
        ax.set_title(label, color="white", fontsize=10)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_xlim(0, 8)
        ax.set_ylim(0, 4)
        for cx, cy in runners:
            ax.add_patch(
                Rectangle(
                    (cx - HW, cy - HH),
                    2 * HW,
                    2 * HH,
                    color="white",
                    fill=False,
                    lw=1.5,
                )
            )
        ax.streamplot(
            x_s,
            y_s,
            frames[idx]["u"].T,
            frames[idx]["v"].T,
            color=frames[idx]["velocity"].T,
            cmap="inferno",
            linewidth=0.8,
            density=1.2,
            arrowstyle="->",
            arrowsize=0.5,
        )
    fig2.suptitle(
        f"Wake Comparison — Step {frames_a[idx]['step']}  "
        f"(t = {frames_a[idx]['step'] * DT:.1f}s)",
        color="white",
        fontsize=14,
        y=0.98,
    )


anim2 = FuncAnimation(fig2, animate, frames=n_frames, interval=80, blit=False)
anim2.save(str(ASSETS / "wake_streamlines.mp4"), writer="ffmpeg", fps=12, dpi=150)
print("Saved assets/wake_streamlines.mp4")

HTML(anim2.to_jshtml())

## 4. The Aerodynamics of Tactical Soccer Positioning

### Three Case Studies in Player Motion

In open play, a soccer player's position relative to teammates and opponents determines not only tactical advantage but also the aerodynamic forces they experience. I modeled three scenarios using 2D CFD with elliptical player cross-sections at a sprint speed of 10 m/s.

#### Case 1: The Isolated Winger (Baseline)

An attacker sprinting into open space faces the full force of the oncoming air. A stagnation zone forms on their chest (high pressure), while the boundary layer separates along the torso flanks, creating a wide, turbulent wake behind them. This establishes the **baseline drag** that all other formations are compared against.

#### Case 2: The Midfield Press Wall (Defensive Phase)

Three defenders aligned perpendicular to the attacking run create a continuous aerodynamic barrier. The individual stagnation regions merge into a single high-pressure wall, and the wakes coalesce into one massive low-velocity zone downstream. This is **aerodynamic choking**, the defensive block deforms the flow field at a macroscopic scale.

#### Case 3: The Overlapping Fullback (Transitional Phase)

A trailing fullback positions themselves directly behind the winger, inside their wake. Because the relative airspeed hitting the fullback is drastically reduced (the winger has already absorbed the stagnation pressure), the fullback experiences significantly lower drag. This is **aerodynamic drafting**, quantified as a percentage reduction from the baseline.

#### Visualisation Legend

| Panel | Case | Plot Type | What to Look For |
|-------|------|-----------|------------------|
| Left | 1 — Winger | Velocity + streamlines | Wide wake, stagnation on front face |
| Centre | 2 — Press Wall | Vorticity (∇ × u) | Merged shear layers, large-scale eddies |
| Right | 3 — Fullback | Pressure + scalar | Low-pressure pocket around trailing player |

#### Setup

- **Grid**: 256 × 192 cells, 8 × 6 unit domain (tall enough for vertical stack)
- **Player shape**: Elliptical `geom.Box` 0.3 × 0.44 units (torso cross-section)
- **Inflow**: uniform 10 m/s from left boundary
- **Re ≈ 3 × 10⁵** (subcritical regime; coarse-grid effective Re ≈ 10³ captures relative wake physics)
- **Solver**: ΦFlow incompressible Navier-Stokes, 400 steps × Δt = 0.03

In [ ]:
"""Module 4 — Three tactical soccer positioning simulations with drag and scalar tracer."""

# ── Domain & boundary conditions ──
BOX4 = geom.Box(x=8.0, y=6.0)
BOUNDARY4 = {
    "x-": vec(x=10.0, y=0.0),  # inflow at sprint speed
    "x+": ZERO_GRADIENT,
    "y": 0,  # slip walls
}

# ── Player shape (elliptical torso cross-section) ──
HW4, HH4 = 0.15, 0.22  # half-width (streamwise), half-height (spanwise)


def player_at(cx, cy):
    return geom.Box(x=(cx - HW4, cx + HW4), y=(cy - HH4, cy + HH4))


# ── Case definitions ──
# Case 1: Isolated winger (baseline)
case1 = [Obstacle(player_at(2.0, 3.0), velocity=vec(x=0.0, y=0.0))]

# Case 2: Midfield press wall — vertical stack, tight gaps
case2 = [
    Obstacle(player_at(2.0, 1.5), velocity=vec(x=0.0, y=0.0)),
    Obstacle(player_at(2.0, 3.0), velocity=vec(x=0.0, y=0.0)),
    Obstacle(player_at(2.0, 4.5), velocity=vec(x=0.0, y=0.0)),
]

# Case 3: Overlapping fullback — in-line, 2-diameter gap
case3 = [
    Obstacle(player_at(2.0, 3.0), velocity=vec(x=0.0, y=0.0)),  # winger (lead)
    Obstacle(player_at(4.0, 3.0), velocity=vec(x=0.0, y=0.0)),  # fullback (trail)
]

# ── Drag masks (front/rear face pressure integration) ──
x_g4 = np.linspace(0, 8.0, 248)
y_g4 = np.linspace(0, 6.0, 186)
XX4, YY4 = np.meshgrid(x_g4, y_g4, indexing="ij")


def face_drag_mask(cx, cy):
    front = (
        (XX4 > cx - HW4 * 1.5)
        & (XX4 < cx - HW4 * 0.8)
        & (YY4 > cy - HH4)
        & (YY4 < cy + HH4)
    )
    rear = (
        (XX4 > cx + HW4 * 0.8)
        & (XX4 < cx + HW4 * 1.5)
        & (YY4 > cy - HH4)
        & (YY4 < cy + HH4)
    )
    return front, rear


def box_drag(p_np, cx, cy):
    front, rear = face_drag_mask(cx, cy)
    return (
        float(p_np[front].mean() - p_np[rear].mean())
        if front.sum() and rear.sum()
        else 0.0
    )


# ── Centred grid for velocity resampling ──
_centered4 = CenteredGrid(0, x=248, y=186, bounds=BOX4)

# ── Solver parameters ──
DT4 = 0.03
N_STEPS4 = 200
SAVE_EVERY4 = 4


def inject_scalar_near_inlet(scalar):
    """Re-inject passive scalar at the inlet boundary."""
    s_np = np.array(scalar.values.native("x,y"))
    s_np[:5, :] = 1.0  # leftmost 5 columns
    return CenteredGrid(
        math.tensor(s_np, math.spatial("x,y")), bounds=BOX4, extrapolation=ZERO_GRADIENT
    )


def run_case(obstacles, label, drag_pos=None, with_scalar=False):
    v0 = StaggeredGrid((10.0, 0.0), BOUNDARY4, x=248, y=186, bounds=BOX4)
    v, _ = fluid.make_incompressible(v0, ())

    scalar = None
    if with_scalar:
        s_init = np.zeros((248, 186))
        s_init[:5, :] = 1.0
        scalar = CenteredGrid(
            math.tensor(s_init, math.spatial("x,y")),
            bounds=BOX4,
            extrapolation=ZERO_GRADIENT,
        )

    frames = []
    drag_series = []
    v = v0

    for i in trange(N_STEPS4, desc=label):
        v = advect.semi_lagrangian(v, v, dt=DT4)
        v, p = fluid.make_incompressible(v, obstacles)

        if with_scalar and scalar is not None:
            scalar = advect.semi_lagrangian(scalar, v, dt=DT4)
            scalar = inject_scalar_near_inlet(scalar)

        if i % SAVE_EVERY4 == 0:
            vc = v @ _centered4
            u_c = np.array(vc.vector[0].values.native("x,y"))
            v_c = np.array(vc.vector[1].values.native("x,y"))
            vel_mag = field.vec_length(v)
            vel_np = np.array(vel_mag.values.native("x,y"))
            p_np = np.array(p.values.native("x,y"))
            vort = field.curl(v)
            vort_np = np.array(vort.values.native("x,y"))

            frame = {
                "velocity": vel_np,
                "u": u_c,
                "v": v_c,
                "pressure": p_np,
                "vorticity": vort_np,
                "step": i,
            }
            if with_scalar and scalar is not None:
                frame["scalar"] = np.array(scalar.values.native("x,y"))

            frames.append(frame)

            if drag_pos:
                drag_series.append(box_drag(p_np, *drag_pos))

    avg_drag = np.mean(drag_series[-len(drag_series) // 5 :]) if drag_series else 0.0
    return frames, avg_drag, drag_series


# ── Execute ──
print("Case 1 — Isolated winger (baseline)")
frames_c1, drag_c1, _ = run_case(case1, "Case 1 (winger)")

print("\nCase 2 — Midfield press wall")
frames_c2, _, _ = run_case(case2, "Case 2 (press wall)")

print("\nCase 3 — Overlapping fullback (with scalar tracer)")
frames_c3, drag_c3, _ = run_case(
    case3, "Case 3 (fullback)", (4.0, 3.0), with_scalar=True
)

# ── Sync frame counts ──
n_frames4 = min(len(frames_c1), len(frames_c2), len(frames_c3))
trim4 = lambda f: f[:n_frames4]
frames_c1, frames_c2, frames_c3 = trim4(frames_c1), trim4(frames_c2), trim4(frames_c3)

drag_ratio = drag_c3 / drag_c1 if drag_c1 != 0 else 0.0
pct_reduction = (1 - drag_ratio) * 100

print(f"\n{'=' * 50}")
print(f"Baseline drag (Case 1 — isolated winger):       {drag_c1:.4f}")
print(f"Trailing drag (Case 3 — overlapping fullback):   {drag_c3:.4f}")
print(f"Drag ratio:                                      {drag_ratio:.3f}")
print(f"Drag reduction:                                  {pct_reduction:.1f}%")
print(f"{'=' * 50}")

In [ ]:
"""Module 4 — Three-panel tactical positioning animation + MP4 export."""

# ── Precompute global vmin/vmax for each panel ──
vmin_v, vmax_v = 0, 12
vort_all = np.concatenate([f["vorticity"].ravel() for f in frames_c2])
vmin_w, vmax_w = np.percentile(vort_all, 2), np.percentile(vort_all, 98)
p_all = np.concatenate([f["pressure"].ravel() for f in frames_c3])
vmin_p, vmax_p = p_all.min(), p_all.max()

# ── 3-panel figure ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), facecolor="#111111")
fig.suptitle(
    "Tactical Soccer Positioning — Aerodynamic Comparison",
    color="white",
    fontsize=15,
    y=0.98,
)

titles = [
    "Case 1 — Isolated Winger (Baseline)",
    "Case 2 — Midfield Press Wall",
    "Case 3 — Overlapping Fullback",
]
frames_list = [frames_c1, frames_c2, frames_c3]
cmaps = ["inferno", "RdBu_r", "magma"]

# Player positions for each case (cx, cy)
players_list = [
    [(2.0, 3.0)],
    [(2.0, 1.5), (2.0, 3.0), (2.0, 4.5)],
    [(2.0, 3.0), (4.0, 3.0)],
]

imgs = []

for ax, title, frames, cmap, players in zip(
    axes, titles, frames_list, cmaps, players_list
):
    ax.set_facecolor("#111111")
    ax.set_title(title, color="white", fontsize=10)
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.set_xlim(0, 8)
    ax.set_ylim(0, 6)

    # Choose field based on panel
    if frames is frames_c1:
        data = frames[0]["velocity"]
        im = ax.imshow(
            data.T,
            origin="lower",
            cmap=cmap,
            aspect="auto",
            extent=[0, 8, 0, 6],
            vmin=vmin_v,
            vmax=vmax_v,
        )
        # Overlay streamlines for baseline
        strm = ax.streamplot(
            np.linspace(0, 8, 248),
            np.linspace(0, 6, 186),
            frames[0]["u"].T,
            frames[0]["v"].T,
            color="white",
            linewidth=0.6,
            density=1.2,
            arrowsize=0.5,
        )
    elif frames is frames_c2:
        data = frames[0]["vorticity"]
        im = ax.imshow(
            data.T,
            origin="lower",
            cmap=cmap,
            aspect="auto",
            extent=[0, 8, 0, 6],
            vmin=vmin_w,
            vmax=vmax_w,
        )
    else:
        data = frames[0]["pressure"]
        im = ax.imshow(
            data.T,
            origin="lower",
            cmap=cmap,
            aspect="auto",
            extent=[0, 8, 0, 6],
            vmin=vmin_p,
            vmax=vmax_p,
        )
        # Passive scalar contour for case 3
        s = frames[0].get("scalar")
        if s is not None:
            ax.contour(
                np.linspace(0, 8, 248),
                np.linspace(0, 6, 186),
                s.T,
                levels=[0.1, 0.5, 0.9],
                colors="cyan",
                linewidths=0.5,
                alpha=0.4,
            )

    # Player markers (ellipses)
    for cx, cy in players:
        ax.add_patch(
            Ellipse(
                (cx, cy),
                width=HW4 * 2,
                height=HH4 * 2,
                color="white",
                fill=False,
                lw=1.5,
            )
        )

    imgs.append(im)

# Drag reduction annotation
ax3 = axes[2]
ax3.text(
    4.5,
    5.3,
    f"Drag Reduction:\n{pct_reduction:.0f}%",
    color="lime",
    fontsize=13,
    weight="bold",
    bbox=dict(facecolor="#111111", edgecolor="lime", alpha=0.8, pad=4),
)

fig.tight_layout(rect=[0, 0, 1, 0.93])

# Colorbar
cbar_ax = fig.add_axes([0.92, 0.12, 0.012, 0.76])  # [left, bottom, width, height]
cbar = fig.colorbar(imgs[0], cax=cbar_ax)
cbar.set_label("Velocity (m/s)", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")


# ── Animation update ──
def update(idx):
    for ax, frames in zip(axes, frames_list):
        ax.clear()
        ax.set_facecolor("#111111")
        ax.set_xlim(0, 8)
        ax.set_ylim(0, 6)
        ax.set_xlabel("x (m)")
        ax.set_ylabel("y (m)")

    # Panel 1 — velocity + streamlines
    ax = axes[0]
    ax.set_title(titles[0], color="white", fontsize=10)
    imgs[0].set_data(frames_c1[idx]["velocity"].T)
    ax.imshow(
        frames_c1[idx]["velocity"].T,
        origin="lower",
        cmap="inferno",
        aspect="auto",
        extent=[0, 8, 0, 6],
        vmin=vmin_v,
        vmax=vmax_v,
    )
    ax.streamplot(
        np.linspace(0, 8, 248),
        np.linspace(0, 6, 186),
        frames_c1[idx]["u"].T,
        frames_c1[idx]["v"].T,
        color="white",
        linewidth=0.6,
        density=1.2,
        arrowsize=0.5,
    )
    for cx, cy in players_list[0]:
        ax.add_patch(
            Ellipse(
                (cx, cy),
                width=HW4 * 2,
                height=HH4 * 2,
                color="white",
                fill=False,
                lw=1.5,
            )
        )

    # Panel 2 — vorticity
    ax = axes[1]
    ax.set_title(titles[1], color="white", fontsize=10)
    imgs[1].set_data(frames_c2[idx]["vorticity"].T)
    ax.imshow(
        frames_c2[idx]["vorticity"].T,
        origin="lower",
        cmap="RdBu_r",
        aspect="auto",
        extent=[0, 8, 0, 6],
        vmin=vmin_w,
        vmax=vmax_w,
    )
    for cx, cy in players_list[1]:
        ax.add_patch(
            Ellipse(
                (cx, cy),
                width=HW4 * 2,
                height=HH4 * 2,
                color="white",
                fill=False,
                lw=1.5,
            )
        )

    # Panel 3 — pressure + scalar
    ax = axes[2]
    ax.set_title(titles[2], color="white", fontsize=10)
    imgs[2].set_data(frames_c3[idx]["pressure"].T)
    ax.imshow(
        frames_c3[idx]["pressure"].T,
        origin="lower",
        cmap="magma",
        aspect="auto",
        extent=[0, 8, 0, 6],
        vmin=vmin_p,
        vmax=vmax_p,
    )
    s = frames_c3[idx].get("scalar")
    if s is not None:
        ax.contour(
            np.linspace(0, 8, 248),
            np.linspace(0, 6, 186),
            s.T,
            levels=[0.1, 0.5, 0.9],
            colors="cyan",
            linewidths=0.5,
            alpha=0.4,
        )
    for cx, cy in players_list[2]:
        ax.add_patch(
            Ellipse(
                (cx, cy),
                width=HW4 * 2,
                height=HH4 * 2,
                color="white",
                fill=False,
                lw=1.5,
            )
        )
    ax.text(
        4.5,
        5.3,
        f"Drag Reduction:\n{pct_reduction:.0f}%",
        color="lime",
        fontsize=13,
        weight="bold",
        bbox=dict(facecolor="#111111", edgecolor="lime", alpha=0.8, pad=4),
    )

    fig.suptitle(
        f"Tactical Soccer Positioning — Step {frames_c1[idx]['step']}  "
        f"(t = {frames_c1[idx]['step'] * DT4:.2f}s)",
        color="white",
        fontsize=14,
        y=0.98,
    )
    return imgs


anim = FuncAnimation(fig, update, frames=n_frames4, interval=80, blit=False)

path = ASSETS / "soccer_positioning.mp4"
anim.save(str(path), writer="ffmpeg", fps=10, dpi=150)
print(f"Saved {path}")

HTML(anim.to_jshtml())

## Macro stress field

In [ ]:
"""Part 3 — The Tactical Continuum: 7v7 Macro Stress Field."""

# ── Pitch & grid ──
PITCH_W, PITCH_H = 10.0, 7.0
NX, NY = 248, 186
x = np.linspace(0, PITCH_W, NX)
y = np.linspace(0, PITCH_H, NY)
XX, YY = np.meshgrid(x, y, indexing="ij")

N_FRAMES = 60
SIGMA_X, SIGMA_Y = 0.50, 0.35  # base gaussian spread (streamwise, spanwise)
VEL_STRETCH = 0.12  # how much velocity stretches the gaussian forward

# ── Synthetic 7v7 trajectories ──
# Attacker start/end: deep build-up → advance and spread
attack_start = np.array(
    [
        [2.0, 3.5],  # CF
        [1.5, 5.2],  # LW
        [1.5, 1.8],  # RW
        [1.2, 4.2],  # CM (left)
        [1.2, 2.8],  # CM (right)
        [0.5, 5.8],  # LB
        [0.5, 1.2],  # RB
    ]
)
attack_end = np.array(
    [
        [6.0, 3.5],  # CF
        [5.8, 5.8],  # LW
        [5.8, 1.2],  # RW
        [4.5, 4.5],  # CM
        [4.5, 2.5],  # CM
        [2.5, 5.8],  # LB
        [2.5, 1.2],  # RB
    ]
)

# Defender start/end: high compact block → drop back
defend_start = np.array(
    [
        [6.0, 3.5],  # CB1
        [6.0, 4.8],  # CB2
        [6.0, 2.2],  # CB3
        [6.8, 3.5],  # DM
        [6.8, 5.5],  # LM
        [6.8, 1.5],  # RM
        [7.8, 3.5],  # SW
    ]
)
defend_end = np.array(
    [
        [4.2, 3.5],
        [4.2, 4.5],
        [4.2, 2.5],
        [5.0, 3.5],
        [5.0, 5.2],
        [5.0, 1.8],
        [6.2, 3.5],
    ]
)

# Ease-out sine for natural acceleration
t_frac = np.linspace(0, 1, N_FRAMES)
eased = np.sin(np.pi * t_frac / 2)

attack_pos = (
    attack_start[np.newaxis]
    + eased[:, np.newaxis, np.newaxis] * (attack_end - attack_start)[np.newaxis]
)
defend_pos = (
    defend_start[np.newaxis]
    + eased[:, np.newaxis, np.newaxis] * (defend_end - defend_start)[np.newaxis]
)

# Velocities (central difference, frame⁻¹ units)
attack_vel = np.gradient(attack_pos, axis=0)
defend_vel = np.gradient(defend_pos, axis=0)


# ── Gaussian with velocity-stretched anisotropic covariance ──
def gaussian_stress(xx, yy, cx, cy, vx, vy, sx_base, sy_base, stretch):
    speed = np.sqrt(vx**2 + vy**2) + 1e-8
    theta = np.arctan2(vy, vx)
    cos_t, sin_t = np.cos(theta), np.sin(theta)

    sx = sx_base + stretch * speed
    sy = sy_base

    a = cos_t**2 / sx**2 + sin_t**2 / sy**2
    b = sin_t * cos_t * (1 / sx**2 - 1 / sy**2)
    c = sin_t**2 / sx**2 + cos_t**2 / sy**2

    dx = xx - cx
    dy = yy - cy
    return np.exp(-0.5 * (a * dx**2 + 2 * b * dx * dy + c * dy**2))


# ── Precompute all frames ──
print("Computing stress fields...")
fields = []
for frame in trange(N_FRAMES):
    attack_field = np.zeros((NX, NY))
    defend_field = np.zeros((NX, NY))

    for p in range(7):
        ax, ay = attack_pos[frame, p, 0], attack_pos[frame, p, 1]
        avx, avy = attack_vel[frame, p, 0], attack_vel[frame, p, 1]
        attack_field += gaussian_stress(
            XX, YY, ax, ay, avx, avy, SIGMA_X, SIGMA_Y, VEL_STRETCH
        )

        dx, dy = defend_pos[frame, p, 0], defend_pos[frame, p, 1]
        dvx, dvy = defend_vel[frame, p, 0], defend_vel[frame, p, 1]
        defend_field += gaussian_stress(
            XX, YY, dx, dy, dvx, dvy, SIGMA_X, SIGMA_Y, VEL_STRETCH
        )

    stress = np.tanh((attack_field - defend_field) / 3.0)
    fields.append(stress)


# ── 3-panel setup (static + stress + stream) ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5), facecolor="#111111")
fig.suptitle(
    "The Tactical Continuum — 7v7 Macro Stress Field",
    color="white",
    fontsize=15,
    y=0.98,
)

vmin, vmax = -1.0, 1.0
# Panel 1: reference positions
ax0 = axes[0]
ax0.set_facecolor("#111111")
ax0.set_title("Initial Positioning (Frame 0)", color="white", fontsize=10)
ax0.set_xlim(0, PITCH_W)
ax0.set_ylim(0, PITCH_H)
ax0.set_aspect("equal")
ax0.scatter(
    attack_pos[0, :, 0],
    attack_pos[0, :, 1],
    c="red",
    s=40,
    label="Attack",
    edgecolors="white",
)
ax0.scatter(
    defend_pos[0, :, 0],
    defend_pos[0, :, 1],
    c="blue",
    s=40,
    label="Defend",
    edgecolors="white",
)
ax0.legend(loc="upper right", facecolor="#111111", labelcolor="white")

# Panel 2: stress contour + zero line
ax1 = axes[1]
ax1.set_facecolor("#111111")
ax1.set_title("Tactical Stress Field", color="white", fontsize=10)
ax1.set_xlabel("x (m)")
ax1.set_ylabel("y (m)")
ax1.set_xlim(0, PITCH_W)
ax1.set_ylim(0, PITCH_H)
ax1.set_aspect("equal")

contour = ax1.contourf(
    x, y, fields[0].T, levels=32, cmap="RdBu_r", vmin=vmin, vmax=vmax
)
zero_line = ax1.contour(x, y, fields[0].T, levels=[0.0], colors="white", linewidths=2.0)

# Panel 3: stress + quiver overlay
ax2 = axes[2]
ax2.set_facecolor("#111111")
ax2.set_title("Player Vectors & Stress", color="white", fontsize=10)
ax2.set_xlabel("x (m)")
ax2.set_ylabel("y (m)")
ax2.set_xlim(0, PITCH_W)
ax2.set_ylim(0, PITCH_H)
ax2.set_aspect("equal")

contour2 = ax2.contourf(
    x, y, fields[0].T, levels=32, cmap="RdBu_r", vmin=vmin, vmax=vmax
)
zero_line2 = ax2.contour(
    x, y, fields[0].T, levels=[0.0], colors="white", linewidths=2.0
)
aq = ax2.quiver(
    attack_pos[0, :, 0],
    attack_pos[0, :, 1],
    attack_vel[0, :, 0],
    attack_vel[0, :, 1],
    color="red",
    scale=1.0,
    width=0.015,
    alpha=0.8,
)
dq = ax2.quiver(
    defend_pos[0, :, 0],
    defend_pos[0, :, 1],
    defend_vel[0, :, 0],
    defend_vel[0, :, 1],
    color="blue",
    scale=1.0,
    width=0.015,
    alpha=0.8,
)

fig.tight_layout(rect=[0, 0, 1, 0.93])
cbar_ax = fig.add_axes([0.92, 0.12, 0.012, 0.76])
cbar = fig.colorbar(contour, cax=cbar_ax)
cbar.set_label("Stress (Attack − Defence)", color="white")
cbar.ax.yaxis.set_tick_params(color="white")
plt.setp(plt.getp(cbar.ax.axes, "yticklabels"), color="white")


# ── Animation ──
def update(idx):
    for ax in axes:
        ax.clear()
        ax.set_facecolor("#111111")
        ax.set_xlim(0, PITCH_W)
        ax.set_ylim(0, PITCH_H)
        ax.set_aspect("equal")

    # Panel 1 — player positions
    ax0 = axes[0]
    ax0.set_title(f"Player Positions (Frame {idx})", color="white", fontsize=10)
    ax0.scatter(
        attack_pos[idx, :, 0],
        attack_pos[idx, :, 1],
        c="red",
        s=50,
        edgecolors="white",
        zorder=5,
    )
    ax0.scatter(
        defend_pos[idx, :, 0],
        defend_pos[idx, :, 1],
        c="blue",
        s=50,
        edgecolors="white",
        zorder=5,
    )

    # Panel 2 — stress contour
    ax1 = axes[1]
    ax1.set_title("Tactical Stress Field", color="white", fontsize=10)
    ax1.set_xlabel("x (m)")
    ax1.set_ylabel("y (m)")
    ax1.contourf(x, y, fields[idx].T, levels=32, cmap="RdBu_r", vmin=vmin, vmax=vmax)
    ax1.contour(x, y, fields[idx].T, levels=[0.0], colors="white", linewidths=2.0)

    # Panel 3 — stress + quiver
    ax2 = axes[2]
    ax2.set_title("Player Vectors & Stress", color="white", fontsize=10)
    ax2.set_xlabel("x (m)")
    ax2.set_ylabel("y (m)")
    ax2.contourf(x, y, fields[idx].T, levels=32, cmap="RdBu_r", vmin=vmin, vmax=vmax)
    ax2.contour(x, y, fields[idx].T, levels=[0.0], colors="white", linewidths=2.0)
    ax2.quiver(
        attack_pos[idx, :, 0],
        attack_pos[idx, :, 1],
        attack_vel[idx, :, 0],
        attack_vel[idx, :, 1],
        color="red",
        scale=1.0,
        width=0.015,
        alpha=0.8,
    )
    ax2.quiver(
        defend_pos[idx, :, 0],
        defend_pos[idx, :, 1],
        defend_vel[idx, :, 0],
        defend_vel[idx, :, 1],
        color="blue",
        scale=1.0,
        width=0.015,
        alpha=0.8,
    )

    fig.suptitle(
        f"The Tactical Continuum — Frame {idx}/60  (t = {idx * 0.1:.1f}s)",
        color="white",
        fontsize=14,
        y=0.98,
    )
    return axes


anim = FuncAnimation(fig, update, frames=N_FRAMES, interval=100, blit=False)

path = ASSETS / "tactical_continuum.mp4"
anim.save(str(path), writer="ffmpeg", fps=10, dpi=150)
print(f"Saved {path}")

HTML(anim.to_jshtml())